# GPU Validation: KV Cache Tier Persistence (TinyLlama-1.1B)

Reruns the end-to-end TinyLlama cache save/restore validation on a GPU, producing
the GPU-side numbers for the paper's break-even analysis (the CPU results already
committed show a 48x TTFT speedup; the honest question is what survives on real
inference hardware).

**Before running:** `Runtime` → `Change runtime type` → select **T4 GPU** → Save.
Then `Runtime` → `Run all` (~10–15 min). The last cell downloads
`tinyllama_gpu_results.json` — that file is the deliverable.

In [ ]:
# Confirm a GPU is attached
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: set Runtime -> Change runtime type -> T4 GPU, then rerun"
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the repo and install (torch/transformers are preinstalled on Colab)
!git clone -q https://github.com/aditya-lawankar/kv-cache-tier-persistence.git
%cd kv-cache-tier-persistence
!pip install -q -e . 2>&1 | tail -1

In [ ]:
# Run 1: 256- and 512-token prompts, 3 trials each
!python benchmarks/tinyllama_integration.py --max-tokens 512 --trials 3 --output benchmarks/results/gpu_512

In [ ]:
# Run 2: ~1800-token prompts (near TinyLlama's 2048 context limit incl. generation)
!python benchmarks/tinyllama_integration.py --max-tokens 1800 --trials 3 --output benchmarks/results/gpu_1800

In [ ]:
# Merge, summarize, and download the deliverable
import json, glob

rows = []
for path in sorted(glob.glob('benchmarks/results/gpu_*/tinyllama_integration_results.json')):
    rows += json.load(open(path))
rows.sort(key=lambda r: r['prompt_tokens'])

with open('tinyllama_gpu_results.json', 'w') as f:
    json.dump(rows, f, indent=2)

print(f"{'tokens':>7} {'cold TTFT':>10} {'warm TTFT':>10} {'speedup':>8} {'save':>7} {'load':>7} {'match':>6} {'device':>7}")
for r in rows:
    print(f"{r['prompt_tokens']:>7} {r['cold_start_ttft_ms']:>8.0f}ms {r['warm_start_ttft_ms']:>8.0f}ms "
          f"{r['speedup_x']:>7.2f}x {r['save_latency_ms']:>5.0f}ms {r['load_latency_ms']:>5.0f}ms "
          f"{'  yes' if r['semantic_match'] else '   NO'} {r.get('device','?'):>7}")

from google.colab import files
files.download('tinyllama_gpu_results.json')